# PhishGuard — Phishing URL Detector Training Pipeline

**Goal:** train a defensive binary classifier that takes a URL string and predicts:

- `0 = Likely legitimate`
- `1 = Potential phishing`

This notebook is designed for Google Colab and produces a single deployable artifact: `phishing_url_detector.joblib`. It combines **character-level URL patterns** with **engineered lexical/domain features**.

### Important dataset decision
The uploaded `url1.csv` is a PhishTank-derived dataset containing verified phishing records only. It does **not** contain a legitimate class, so it is not used as the sole training set. Instead, the notebook trains on the UCI PhiUSIIL dataset, which contains both legitimate and phishing URLs, and then uses your uploaded PhishTank file as an additional external phishing generalization check. The UCI dataset reports 134,850 legitimate and 100,945 phishing URLs.

### Security design
The detector analyzes the URL **as text only**. It does not open, crawl, redirect to, or execute the submitted website. This avoids turning the API into a server-side URL fetcher.

### Methodology
1. Load and inspect data.
2. Clean and deduplicate URLs.
3. Create a domain-aware train/validation/test split.
4. Engineer lexical/domain features.
5. Add character n-gram features with hashing.
6. Tune the SVM regularization parameter.
7. Select a classification threshold using validation F1.
8. Evaluate on an unseen test set.
9. Evaluate separately on your uploaded PhishTank records.
10. Save the full preprocessing + model artifact for Render and the Chrome extension.


## 0. Install the exact training dependencies

Run this cell first. Pinning versions reduces the chance that the serialized model behaves differently between Colab and Render.

In [ ]:
!pip -q install pandas==2.2.3 numpy==1.26.4 scipy==1.13.1 scikit-learn==1.5.2 joblib==1.4.2 ucimlrepo==0.0.7 matplotlib==3.9.4 seaborn==0.13.2

## 1. Imports and reproducibility

In [ ]:
from __future__ import annotations

import json
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import sparse
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.metrics import (
    accuracy_score, average_precision_score, classification_report,
    confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score,
    precision_recall_curve, roc_curve
)
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.pipeline import FeatureUnion
from sklearn.preprocessing import MaxAbsScaler
from sklearn.svm import LinearSVC
from ucimlrepo import fetch_ucirepo

SEED = 42
np.random.seed(SEED)

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('pandas:', pd.__version__)


## 2. Create the shared URL feature module

This is critical. The final `.joblib` contains a custom transformer. The same `url_features.py` file must be present in the Render application so that `joblib.load()` can reconstruct the pipeline.

In [ ]:
from pathlib import Path

FEATURE_MODULE = Path('url_features.py')
MODULE_SOURCE = 'from __future__ import annotations\n\nimport ipaddress\nimport math\nimport re\nfrom collections import Counter\nfrom urllib.parse import urlsplit\n\nimport numpy as np\nfrom scipy import sparse\nfrom sklearn.base import BaseEstimator, TransformerMixin\n\nSUSPICIOUS_TLDS = {\n    "zip", "mov", "top", "xyz", "click", "link", "work", "live", "online",\n    "site", "shop", "club", "icu", "buzz", "rest", "fit", "cfd", "gq", "tk",\n    "ml", "ga", "cf", "win", "loan", "download", "stream", "pro", "support",\n}\n\nSUSPICIOUS_KEYWORDS = {\n    "login", "log-in", "signin", "sign-in", "verify", "verification", "secure",\n    "security", "account", "update", "confirm", "confirmation", "password", "passwd",\n    "credential", "wallet", "payment", "billing", "invoice", "bank", "banking", "paypal",\n    "amazon", "microsoft", "apple", "google", "facebook", "instagram", "netflix",\n    "dhl", "fedex", "ups", "irs", "crypto", "bitcoin", "bonus", "gift", "reward",\n    "free", "urgent", "suspended", "unlock", "recover", "support", "helpdesk", "otp",\n}\n\nBRAND_KEYWORDS = {\n    "google", "microsoft", "apple", "amazon", "paypal", "facebook", "instagram", "netflix",\n    "linkedin", "twitter", "x.com", "dropbox", "adobe", "dhl", "fedex", "ups", "spotify",\n    "steam", "binance", "coinbase", "bank", "visa", "mastercard", "amex",\n}\n\n\ndef _safe_url(url: str) -> str:\n    return str(url).strip()\n\n\ndef _parse(url: str):\n    raw = _safe_url(url)\n    candidate = raw if re.match(r"^[a-zA-Z][a-zA-Z0-9+.-]*://", raw) else "http://" + raw\n    try:\n        return urlsplit(candidate)\n    except ValueError:\n        return urlsplit("http://invalid.local")\n\n\ndef _is_ip(hostname: str) -> int:\n    host = hostname.strip("[]")\n    try:\n        ipaddress.ip_address(host)\n        return 1\n    except ValueError:\n        return 0\n\n\ndef _entropy(text: str) -> float:\n    if not text:\n        return 0.0\n    counts = Counter(text)\n    n = len(text)\n    return float(-sum((c / n) * math.log2(c / n) for c in counts.values()))\n\n\ndef extract_url_features(url: str) -> list[float]:\n    raw = _safe_url(url)\n    parts = _parse(raw)\n    host = (parts.hostname or "").lower()\n    path = parts.path or ""\n    query = parts.query or ""\n    fragment = parts.fragment or ""\n    full_lower = raw.lower()\n\n    length = len(raw)\n    letters = sum(ch.isalpha() for ch in raw)\n    digits = sum(ch.isdigit() for ch in raw)\n    special = sum(not ch.isalnum() for ch in raw)\n    host_digits = sum(ch.isdigit() for ch in host)\n    path_segments = len([p for p in path.split("/") if p])\n    query_params = 0 if not query else query.count("&") + 1\n    subdomains = max(0, len([p for p in host.split(".") if p]) - 2)\n    tld = host.rsplit(".", 1)[-1] if "." in host else ""\n    keyword_hits = sum(1 for k in SUSPICIOUS_KEYWORDS if k in full_lower)\n    brand_hits = sum(1 for k in BRAND_KEYWORDS if k in full_lower)\n    percent_encoded = len(re.findall(r"%[0-9a-fA-F]{2}", raw))\n    repeated_separator = int(bool(re.search(r"[._-]{2,}", raw)))\n\n    features = [\n        length,\n        len(host),\n        len(path),\n        len(query),\n        len(fragment),\n        len(parts.scheme),\n        raw.count("."),\n        raw.count("-"),\n        raw.count("_"),\n        raw.count("/"),\n        raw.count("?"),\n        raw.count("="),\n        raw.count("&"),\n        raw.count("@"),\n        raw.count("%"),\n        raw.count("#"),\n        raw.count(":"),\n        raw.count("\\\\"),\n        raw.count(";") ,\n        raw.count("+"),\n        raw.count(","),\n        digits,\n        letters,\n        special,\n        host_digits,\n        sum(1 for ch in path if ch.isdigit()),\n        sum(1 for ch in query if ch.isdigit()),\n        digits / max(1, length),\n        letters / max(1, length),\n        special / max(1, length),\n        host_digits / max(1, len(host)),\n        int(parts.scheme.lower() == "https"),\n        int(parts.scheme.lower() == "http"),\n        int(_is_ip(host)),\n        int("xn--" in host.lower()),\n        int("@" in raw),\n        int(bool(parts.port)),\n        subdomains,\n        len(tld),\n        int(tld in SUSPICIOUS_TLDS),\n        path_segments,\n        query_params,\n        keyword_hits,\n        brand_hits,\n        percent_encoded,\n        repeated_separator,\n        _entropy(raw),\n        _entropy(host),\n        int(bool(re.search(r"//.+//", raw.replace("https://", "").replace("http://", "")))),\n    ]\n    return [float(x) for x in features]\n\n\ndef explain_url(url: str) -> list[str]:\n    raw = _safe_url(url)\n    parts = _parse(raw)\n    host = (parts.hostname or "").lower()\n    full_lower = raw.lower()\n    tld = host.rsplit(".", 1)[-1] if "." in host else ""\n    signals: list[str] = []\n\n    if len(raw) > 120:\n        signals.append("URL is unusually long")\n    if len(host) > 45:\n        signals.append("hostname is unusually long")\n    if raw.count(".") >= 5:\n        signals.append("many subdomain/domain separators")\n    if raw.count("-") >= 4:\n        signals.append("many hyphens")\n    if raw.count("@") > 0:\n        signals.append("contains @ in the URL")\n    if "xn--" in host:\n        signals.append("contains punycode hostname")\n    if _is_ip(host):\n        signals.append("uses an IP address instead of a normal domain")\n    if tld in SUSPICIOUS_TLDS:\n        signals.append(f"uses a TLD often seen in suspicious URL datasets (.{tld})")\n    keyword_matches = [k for k in SUSPICIOUS_KEYWORDS if k in full_lower]\n    if keyword_matches:\n        signals.append("contains security/account-related keywords")\n    brand_matches = [k for k in BRAND_KEYWORDS if k in full_lower]\n    if brand_matches:\n        signals.append("contains a recognizable brand/service name in the URL")\n    if "%" in raw:\n        signals.append("contains percent-encoded characters")\n    if parts.scheme.lower() == "http":\n        signals.append("uses HTTP rather than HTTPS")\n    if len([p for p in host.split(".") if p]) >= 5:\n        signals.append("has multiple hostname levels")\n\n    if not signals:\n        signals.append("no strong lexical warning signal was detected")\n    return signals[:8]\n\n\nclass URLFeatureExtractor(BaseEstimator, TransformerMixin):\n    """Convert a 1-D URL sequence into sparse numeric lexical/security features."""\n\n    def fit(self, X, y=None):\n        return self\n\n    def transform(self, X):\n        rows = [extract_url_features(str(x)) for x in X]\n        matrix = np.asarray(rows, dtype=np.float32)\n        return sparse.csr_matrix(matrix)\n'
FEATURE_MODULE.write_text(MODULE_SOURCE, encoding='utf-8')
print('Wrote:', FEATURE_MODULE.resolve())


## 3. Upload your PhishTank CSV

The file you uploaded in this conversation is named `url1.csv`. When running this notebook in Colab, upload that same file when prompted.

In [ ]:
from google.colab import files

uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))

PHISHTANK_PATH = 'url1.csv'
if not Path(PHISHTANK_PATH).exists():
    # Helpful fallback if the uploaded filename is different.
    csv_candidates = list(Path('.').glob('*.csv'))
    if len(csv_candidates) == 1:
        PHISHTANK_PATH = str(csv_candidates[0])
    else:
        raise FileNotFoundError('Could not find url1.csv. Upload the supplied PhishTank CSV.')

print('Using PhishTank file:', PHISHTANK_PATH)


## 4. Inspect your uploaded dataset before touching it

The expected result is approximately 73k rows, with `url` and `target` columns plus PhishTank metadata. The notebook will explicitly verify the class situation.

In [ ]:
phishtank_raw = pd.read_csv(PHISHTANK_PATH)
print('Shape:', phishtank_raw.shape)
print('Columns:', list(phishtank_raw.columns))
print('Missing values:')
print(phishtank_raw.isna().sum())
print('Exact duplicate rows:', phishtank_raw.duplicated().sum())

if 'url' not in phishtank_raw.columns:
    raise ValueError("The PhishTank file must contain a 'url' column.")

print('Target distribution:')
print(phishtank_raw['target'].value_counts(dropna=False).head(20) if 'target' in phishtank_raw.columns else 'target column not found')


## 5. Load the official UCI PhiUSIIL dataset

PhiUSIIL contains both legitimate and phishing URLs and is appropriate for binary training. Its official UCI page states that label `1` is legitimate and label `0` is phishing. We reverse that mapping for this project so that the positive class is phishing (`1`).

In [ ]:
uci = fetch_ucirepo(id=967)

X_uci = uci.data.features.copy()
y_uci = uci.data.targets.copy()

if isinstance(y_uci, pd.DataFrame):
    if 'label' in y_uci.columns:
        y_uci = y_uci['label']
    else:
        y_uci = y_uci.iloc[:, 0]

uci_df = X_uci.copy()
if 'URL' not in uci_df.columns:
    raise ValueError(f"Unexpected UCI dataset schema. URL column not found. Columns: {list(uci_df.columns)}")

uci_df = uci_df.rename(columns={'URL': 'url'})
uci_df['source_label'] = y_uci.to_numpy()
uci_df['label'] = (uci_df['source_label'].astype(int) == 0).astype(int)

# Keep only the URL and the optional original domain column for grouping.
keep = ['url', 'label'] + (["Domain"] if 'Domain' in uci_df.columns else [])
uci_df = uci_df[keep].copy()

print('UCI shape:', uci_df.shape)
print('Final binary labels:')
print(uci_df['label'].value_counts().sort_index())


## 6. Clean URLs and remove exact duplicates

Do not use PhishTank metadata such as `target`, `verified`, submission date, or `phish_detail_url` as model inputs. Those fields would create leakage or would simply be unavailable for a new URL.

In [ ]:
def normalize_url_for_dedupe(value: str) -> str:
    return str(value).strip().lower()

def valid_url_text(value: str) -> bool:
    s = str(value).strip()
    return 3 <= len(s) <= 4096 and not s.lower().startswith(('javascript:', 'data:', 'file:'))

uci_df['url'] = uci_df['url'].astype(str).str.strip()
uci_df = uci_df[uci_df['url'].map(valid_url_text)].copy()
uci_df['url_norm'] = uci_df['url'].map(normalize_url_for_dedupe)
uci_df = uci_df.drop_duplicates('url_norm').reset_index(drop=True)

print('Clean UCI rows:', len(uci_df))
print(uci_df['label'].value_counts(normalize=True).rename('proportion'))


## 7. Build a domain-aware train / validation / test split

A random row split can overestimate performance when many URLs from the same domain appear in both train and test. We use `StratifiedGroupKFold` so domain groups remain separated as much as possible.

In [ ]:
from urllib.parse import urlsplit

def fallback_host(url: str) -> str:
    candidate = url if '://' in url else 'http://' + url
    try:
        host = (urlsplit(candidate).hostname or '').lower().strip('.')
    except ValueError:
        host = ''
    return host or '__invalid_host__'

if 'Domain' in uci_df.columns:
    groups = uci_df['Domain'].fillna('').astype(str).str.lower().str.strip()
    groups = groups.where(groups.str.len() > 0, uci_df['url'].map(fallback_host))
else:
    groups = uci_df['url'].map(fallback_host)

uci_df['domain_group'] = groups

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
first_split = next(cv.split(uci_df['url'], uci_df['label'], groups=uci_df['domain_group']))
train_val_idx, test_idx = first_split

train_val_df = uci_df.iloc[train_val_idx].reset_index(drop=True)
test_df = uci_df.iloc[test_idx].reset_index(drop=True)

inner_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED + 1)
inner_split = next(inner_cv.split(train_val_df['url'], train_val_df['label'], groups=train_val_df['domain_group']))
train_idx, val_idx = inner_split

train_df = train_val_df.iloc[train_idx].reset_index(drop=True)
val_df = train_val_df.iloc[val_idx].reset_index(drop=True)

for name, part in [('TRAIN', train_df), ('VALIDATION', val_df), ('TEST', test_df)]:
    print(f'{name:12s}: rows={len(part):,}  phishing={int(part.label.sum()):,}  legitimate={int((1-part.label).sum()):,}')

train_groups = set(train_df.domain_group)
val_groups = set(val_df.domain_group)
test_groups = set(test_df.domain_group)
print('Domain group overlap train/val:', len(train_groups & val_groups))
print('Domain group overlap train/test:', len(train_groups & test_groups))
print('Domain group overlap val/test:', len(val_groups & test_groups))


## 8. Build the hybrid feature representation

The model receives two complementary views of every URL:

**A. Character n-grams** — a `HashingVectorizer` learns no vocabulary and captures patterns such as brand typos, unusual separators, encoded strings, and suspicious path structures.

**B. Engineered lexical/security features** — the supplied `URLFeatureExtractor` computes URL length, hostname length, path/query structure, digit and symbol ratios, IP-host usage, punycode, HTTPS/HTTP, subdomain depth, suspicious TLDs, keywords, entropy, and other lexical signals.

Hashing keeps the final model compact and makes Render deployment much easier than storing a huge learned TF-IDF vocabulary.

In [ ]:
from url_features import URLFeatureExtractor, extract_url_features

preprocessor = FeatureUnion(
    transformer_list=[
        ('char_ngrams', HashingVectorizer(
            analyzer='char',
            ngram_range=(3, 5),
            n_features=2**18,
            alternate_sign=False,
            norm='l2',
            lowercase=True,
            dtype=np.float32,
        )),
        ('lexical', URLFeatureExtractor()),
    ],
    n_jobs=1,
)

print('Fitting feature representation on training URLs...')
X_train_raw = preprocessor.fit_transform(train_df['url'].tolist())
X_val_raw = preprocessor.transform(val_df['url'].tolist())
X_test_raw = preprocessor.transform(test_df['url'].tolist())

print('Train feature matrix:', X_train_raw.shape, 'sparse:', sparse.issparse(X_train_raw))
print('Approx train nnz:', X_train_raw.nnz if sparse.issparse(X_train_raw) else 'n/a')


## 9. Scale the mixed sparse features

`MaxAbsScaler` keeps the data sparse while bringing the engineered numeric features onto a compatible scale with the hashed n-gram block.

In [ ]:
scaler = MaxAbsScaler()
X_train = scaler.fit_transform(X_train_raw)
X_val = scaler.transform(X_val_raw)
X_test = scaler.transform(X_test_raw)

y_train = train_df['label'].to_numpy(dtype=np.int8)
y_val = val_df['label'].to_numpy(dtype=np.int8)
y_test = test_df['label'].to_numpy(dtype=np.int8)

print('Scaled train matrix:', X_train.shape)


## 10. Hyperparameter tuning — choose the SVM C value

We do not blindly test dozens of settings. We use a stratified 40,000-row subset of the training split to compare several C values against the **separate validation set**. This keeps computation practical in Colab while still performing real tuning.

In [ ]:
MAX_TUNE_ROWS = min(40_000, len(y_train))
tune_idx, _ = train_test_split(
    np.arange(len(y_train)),
    train_size=MAX_TUNE_ROWS,
    stratify=y_train,
    random_state=SEED,
)

C_VALUES = [0.5, 1.0, 2.0, 4.0]
tuning_results = []

for C in C_VALUES:
    print(f'Training LinearSVC C={C} on {len(tune_idx):,} rows...')
    clf = LinearSVC(
        C=C,
        class_weight='balanced',
        dual=True,
        max_iter=5000,
        tol=1e-4,
        random_state=SEED,
    )
    clf.fit(X_train[tune_idx], y_train[tune_idx])
    val_pred = clf.predict(X_val)
    val_score = clf.decision_function(X_val)
    tuning_results.append({
        'C': C,
        'accuracy': accuracy_score(y_val, val_pred),
        'precision': precision_score(y_val, val_pred, zero_division=0),
        'recall': recall_score(y_val, val_pred, zero_division=0),
        'f1': f1_score(y_val, val_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_val, val_score),
        'pr_auc': average_precision_score(y_val, val_score),
    })

tuning_df = pd.DataFrame(tuning_results).sort_values(['f1', 'recall'], ascending=False).reset_index(drop=True)
display(tuning_df)
best_C = float(tuning_df.loc[0, 'C'])
print('Selected C:', best_C)


## 11. Fit the selected model on the complete training split and choose a decision threshold

The default SVM boundary is `0.0`, but phishing detection is a cost-sensitive problem: missing a phishing URL can be more important than allowing a small number of false alarms. We therefore choose the threshold using validation F1, then lock it before final testing.

In [ ]:
tuned_model = LinearSVC(
    C=best_C,
    class_weight='balanced',
    dual=True,
    max_iter=5000,
    tol=1e-4,
    random_state=SEED,
)
tuned_model.fit(X_train, y_train)

val_scores = tuned_model.decision_function(X_val)
precision_vals, recall_vals, thresholds = precision_recall_curve(y_val, val_scores)

f1_vals = 2 * precision_vals * recall_vals / np.maximum(precision_vals + recall_vals, 1e-12)
if len(thresholds) == 0:
    decision_threshold = 0.0
else:
    best_idx = int(np.nanargmax(f1_vals[:-1]))
    decision_threshold = float(thresholds[best_idx])

val_pred_thresholded = (val_scores >= decision_threshold).astype(int)
print('Selected decision threshold:', decision_threshold)
print('Validation F1:', f1_score(y_val, val_pred_thresholded, zero_division=0))
print('Validation precision:', precision_score(y_val, val_pred_thresholded, zero_division=0))
print('Validation recall:', recall_score(y_val, val_pred_thresholded, zero_division=0))


## 12. Train the final detector on train + validation data

The test set remains untouched until this point. Because the chosen hashed feature representation is stateless, we can safely transform train + validation using the same feature map. The scaler is refit using train + validation, and the classifier is retrained from scratch with the selected C.

In [ ]:
trainval_urls = pd.concat([train_df['url'], val_df['url']], ignore_index=True)
y_trainval = np.concatenate([y_train, y_val]).astype(np.int8)

X_trainval_raw = preprocessor.transform(trainval_urls.tolist())
final_scaler = MaxAbsScaler()
X_trainval = final_scaler.fit_transform(X_trainval_raw)
X_test_final = final_scaler.transform(X_test_raw)

final_model = LinearSVC(
    C=best_C,
    class_weight='balanced',
    dual=True,
    max_iter=5000,
    tol=1e-4,
    random_state=SEED,
)
final_model.fit(X_trainval, y_trainval)

test_scores = final_model.decision_function(X_test_final)
test_pred = (test_scores >= decision_threshold).astype(int)


## 13. Final evaluation on the unseen test set

Report more than accuracy. For a phishing detector, **precision, recall, F1, PR-AUC, ROC-AUC, and the confusion matrix** provide a much more useful picture.

In [ ]:
def evaluate_binary(name, y_true, scores, threshold):
    pred = (scores >= threshold).astype(int)
    metrics = {
        'dataset': name,
        'accuracy': accuracy_score(y_true, pred),
        'precision': precision_score(y_true, pred, zero_division=0),
        'recall': recall_score(y_true, pred, zero_division=0),
        'f1': f1_score(y_true, pred, zero_division=0),
        'roc_auc': roc_auc_score(y_true, scores),
        'pr_auc': average_precision_score(y_true, scores),
    }
    print(json.dumps(metrics, indent=2))
    print('\nClassification report:')
    print(classification_report(y_true, pred, target_names=['Likely legitimate', 'Potential phishing'], zero_division=0))
    print('Confusion matrix:')
    print(confusion_matrix(y_true, pred))
    return metrics, pred

test_metrics, test_pred = evaluate_binary('UCI grouped test', y_test, test_scores, decision_threshold)


## 14. Visual evaluation plots

In [ ]:
cm = confusion_matrix(y_test, test_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Legitimate', 'Phishing'], yticklabels=['Legitimate', 'Phishing'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('PhishGuard Confusion Matrix — Unseen Test Set')
plt.tight_layout()
plt.show()

roc_fpr, roc_tpr, _ = roc_curve(y_test, test_scores)
plt.figure(figsize=(6, 5))
plt.plot(roc_fpr, roc_tpr, label=f"ROC-AUC = {test_metrics['roc_auc']:.4f}")
plt.plot([0, 1], [0, 1], '--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.tight_layout()
plt.show()

pr_p, pr_r, _ = precision_recall_curve(y_test, test_scores)
plt.figure(figsize=(6, 5))
plt.plot(pr_r, pr_p, label=f"PR-AUC = {test_metrics['pr_auc']:.4f}")
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision–Recall Curve')
plt.legend()
plt.tight_layout()
plt.show()


## 15. External check using your uploaded PhishTank records

These records are phishing-only, so this is **not** a complete accuracy test. The useful metric here is phishing recall: how many of the unseen PhishTank URLs are recognized as phishing. Exact URLs already present in the UCI dataset are excluded to reduce contamination.

In [ ]:
pt = phishtank_raw[['url']].copy()
pt['url'] = pt['url'].astype(str).str.strip()
pt = pt[pt['url'].map(valid_url_text)].copy()
pt['url_norm'] = pt['url'].map(normalize_url_for_dedupe)
pt = pt.drop_duplicates('url_norm').reset_index(drop=True)

uci_url_set = set(uci_df['url_norm'])
external_pt = pt[~pt['url_norm'].isin(uci_url_set)].reset_index(drop=True)

print('Uploaded unique phishing URLs:', len(pt))
print('External URLs after exact-overlap removal:', len(external_pt))

if len(external_pt) > 0:
    X_pt_raw = preprocessor.transform(external_pt['url'].tolist())
    X_pt = final_scaler.transform(X_pt_raw)
    pt_scores = final_model.decision_function(X_pt)
    pt_pred = (pt_scores >= decision_threshold).astype(int)
    phishing_recall = float((pt_pred == 1).mean())
    print(f'PhishTank external phishing recall: {phishing_recall:.4f} ({pt_pred.sum():,}/{len(pt_pred):,})')
else:
    phishing_recall = None
    print('No non-overlapping PhishTank URLs remained after exact deduplication.')


## 16. Sanity-check URLs

These are functional checks of the inference pipeline, not ground-truth claims about the live websites. A typosquat-style string such as `y0utube.com` is useful as a test input because it contains a brand-like mutation; whether the model flags it depends on what it learned.

In [ ]:
sanity_urls = [
    'https://example.com/',
    'https://www.google.com/',
    'https://www.youtube.com/',
    'http://y0utube.com/login',
    'https://account-security-example.invalid/verify',
    'https://xn--example-9db.invalid/login',
]

X_sanity_raw = preprocessor.transform(sanity_urls)
X_sanity = final_scaler.transform(X_sanity_raw)
sanity_scores = final_model.decision_function(X_sanity)

sanity_df = pd.DataFrame({
    'url': sanity_urls,
    'decision_score': sanity_scores,
    'prediction': np.where(sanity_scores >= decision_threshold, 'Potential phishing', 'Likely legitimate'),
})
display(sanity_df)


## 17. Save the complete deployable artifact

The saved object includes:
- the feature representation,
- the scaler,
- the trained SVM,
- the locked decision threshold,
- model metadata.

**Do not save only the classifier.** The preprocessing and threshold are part of the model contract.

In [ ]:
MODEL_VERSION = 'phishguard-' + datetime.now(timezone.utc).strftime('%Y.%m.%d')

artifact = {
    'preprocessor': preprocessor,
    'scaler': final_scaler,
    'classifier': final_model,
    'threshold': decision_threshold,
    'model_version': MODEL_VERSION,
    'positive_class': 'Potential phishing',
    'negative_class': 'Likely legitimate',
}

MODEL_PATH = Path('phishing_url_detector.joblib')
METADATA_PATH = Path('model_metadata.json')

metadata = {
    'model_version': MODEL_VERSION,
    'positive_class': 'Potential phishing',
    'negative_class': 'Likely legitimate',
    'decision_threshold': float(decision_threshold),
    'best_C': float(best_C),
    'feature_design': {
        'char_ngrams': {'analyzer': 'char', 'ngram_range': [3, 5], 'n_features': 2**18, 'alternate_sign': False},
        'engineered_feature_count': len(extract_url_features('https://example.com/')),
    },
    'training_data': {
        'primary': 'UCI PhiUSIIL Phishing URL Dataset (ID 967)',
        'uploaded_phishtank_used_for_training': False,
        'uploaded_phishtank_role': 'external phishing generalization check',
    },
    'test_metrics': test_metrics,
    'phishing_external_recall': phishing_recall,
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
}

joblib.dump(artifact, MODEL_PATH, compress=3)
METADATA_PATH.write_text(json.dumps(metadata, indent=2), encoding='utf-8')

print('Saved:', MODEL_PATH.resolve())
print('Model size MB:', MODEL_PATH.stat().st_size / (1024**2))
print('Saved metadata:', METADATA_PATH.resolve())


## 18. Final artifact self-test — run this before downloading

This catches the most important deployment failure: a model that saves successfully but cannot be reloaded and used for inference.

In [ ]:
reloaded = joblib.load(MODEL_PATH)
required_keys = {'preprocessor', 'scaler', 'classifier', 'threshold', 'model_version'}
missing = required_keys - set(reloaded.keys())
if missing:
    raise RuntimeError(f'Model artifact is missing keys: {missing}')

probe_url = 'http://y0utube.com/login'
probe_X = reloaded['preprocessor'].transform([probe_url])
probe_X = reloaded['scaler'].transform(probe_X)
probe_score = float(reloaded['classifier'].decision_function(probe_X)[0])
probe_pred = int(probe_score >= float(reloaded['threshold']))

print('Reload test: PASS')
print('Model version:', reloaded['model_version'])
print('Probe score:', probe_score)
print('Probe prediction:', 'Potential phishing' if probe_pred else 'Likely legitimate')


## 19. Download the model and the required feature module

Download **both** files. The Render deployment needs the model artifact and the exact `url_features.py` implementation used when the artifact was created.

In [ ]:
from google.colab import files

files.download(str(MODEL_PATH))
files.download(str(METADATA_PATH))
files.download('url_features.py')


## 20. Optional: create a ZIP for deployment

After downloading, put these files together:

```text
phishguard-api/
├── app.py
├── url_features.py
├── phishing_url_detector.joblib
├── model_metadata.json
├── requirements.txt
├── .python-version
└── render.yaml
```

The files `app.py`, `requirements.txt`, `.python-version`, `render.yaml`, and `url_features.py` are already provided in the deployment package.

In [ ]:
import zipfile

DEPLOY_FILES = [
    'app.py',
    'url_features.py',
    'phishing_url_detector.joblib',
    'model_metadata.json',
    'requirements.txt',
    '.python-version',
    'render.yaml',
]
missing = [f for f in DEPLOY_FILES if not Path(f).exists()]
if missing:
    print('Missing deployment files:', missing)
else:
    zip_path = Path('phishguard-api-deploy.zip')
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in DEPLOY_FILES:
            zf.write(f, arcname=f)
    print('Created:', zip_path.resolve())
    files.download(str(zip_path))
